<a href="https://colab.research.google.com/github/Safiyanoorain/AI-Music-Generation/blob/main/AI_Music_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install music21

In [ ]:
import tensorflow as tf
import numpy as np

print("TensorFlow version:", tf.__version__)
print("NumPy version:", np.__version__)

TensorFlow version: 2.20.0
NumPy version: 2.1.3


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

project_path = "/content/drive/MyDrive/AI_Music_Generation"

os.makedirs(project_path, exist_ok=True)
os.makedirs(project_path + "/midi_dataset", exist_ok=True)
os.makedirs(project_path + "/generated_music", exist_ok=True)

print("Project folders created successfully!")
print(project_path)

Project folders created successfully!
/content/drive/MyDrive/AI_Music_Generation


In [ ]:
import os

dataset_dir = "/content/drive/MyDrive/AI_Music_Generation/midi_dataset"

print(os.listdir(dataset_dir))

['albeniz', 'bach', 'balakir', 'beeth', 'borodin', 'brahms', 'burgm', 'chopin', 'debussy', 'granados', 'grieg', 'haydn', 'liszt', 'mendelssohn', 'mozart', 'muss', 'schubert', 'schumann', 'tschai', 'archive.zip']


In [ ]:
from music21 import converter, note, chord
import os

# Find the first MIDI file
first_midi = None

for root, folders, files in os.walk(dataset_dir):
    for file in files:
        if file.lower().endswith((".mid", ".midi")):
            first_midi = os.path.join(root, file)
            break
    if first_midi:
        break

print("MIDI file selected:")
print(first_midi)

MIDI file selected:
/content/drive/MyDrive/AI_Music_Generation/midi_dataset/albeniz/alb_esp1.mid


In [ ]:
from music21 import converter, note, chord

midi = converter.parse(first_midi)

notes = []

for element in midi.recurse().notes:
    if isinstance(element, note.Note):
        notes.append(str(element.pitch))

    elif isinstance(element, chord.Chord):
        notes.append(
            '.'.join(str(p.midi) for p in element.pitches)
        )

print("Number of notes/chords:", len(notes))
print("First 20:")
print(notes[:20])

Number of notes/chords: 606
First 20:
['A5', 'E6', 'E6', '86.88', 'D6', 'C6', 'D6', 'E6', 'F6', 'G6', 'E6', 'F6', 'E6', '86.88', 'D6', 'C6', 'B-5', 'A3', 'C#4', 'E4']


In [ ]:
import numpy as np
from tensorflow.keras.utils import to_categorical

# Use first 500 notes to keep the project lightweight
notes = notes[:500]

sequence_length = 20

pitchnames = sorted(set(notes))
note_to_int = {n: i for i, n in enumerate(pitchnames)}

network_input = []
network_output = []

for i in range(len(notes) - sequence_length):
    sequence_in = notes[i:i + sequence_length]
    sequence_out = notes[i + sequence_length]

    network_input.append([note_to_int[x] for x in sequence_in])
    network_output.append(note_to_int[sequence_out])

X = np.array(network_input, dtype=np.float32)
X = np.reshape(X, (X.shape[0], X.shape[1], 1))
X = X / float(len(pitchnames))

y = to_categorical(network_output, num_classes=len(pitchnames))

print("Input shape:", X.shape)
print("Output shape:", y.shape)
print("Number of unique notes/chords:", len(pitchnames))

Input shape: (480, 20, 1)
Output shape: (480, 48)
Number of unique notes/chords: 48


In [ ]:
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Input, LSTM, Dropout, Dense

model = Sequential([
    Input(shape=(X.shape[1], X.shape[2])),
    LSTM(64, return_sequences=True),
    Dropout(0.2),
    LSTM(64),
    Dense(len(pitchnames), activation="softmax")
])

model.compile(
    loss="categorical_crossentropy",
    optimizer="adam"
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 20, 64)         │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 20, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 48)             │         3,120 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 53,040 (207.19 KB)

 Trainable params: 53,040 (207.19 KB)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history = model.fit(
    X,
    y,
    epochs=5,
    batch_size=32,
    verbose=1
)

Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 4s 40ms/step - loss: 3.7864
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 49ms/step - loss: 3.4386
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 44ms/step - loss: 3.3142
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 37ms/step - loss: 3.2907
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 1s 40ms/step - loss: 3.2818


In [ ]:
model_path = project_path + "/music_lstm.keras"

model.save(model_path)

print("Model saved successfully!")
print(model_path)

Model saved successfully!
/content/drive/MyDrive/AI_Music_Generation/music_lstm.keras


In [ ]:
import numpy as np

# Start with a random sequence from our training data
start = np.random.randint(0, len(network_input))
pattern = network_input[start].copy()

prediction_output = []

# Generate 50 new notes/chords
for _ in range(50):
    prediction_input = np.array(pattern, dtype=np.float32)
    prediction_input = prediction_input.reshape(1, sequence_length, 1)
    prediction_input = prediction_input / float(len(pitchnames))

    prediction = model.predict(prediction_input, verbose=0)

    index = np.argmax(prediction)
    result = pitchnames[index]

    prediction_output.append(result)

    pattern = pattern[1:] + [index]

print("Generated notes/chords:", len(prediction_output))
print("First 20:")
print(prediction_output[:20])

Generated notes/chords: 50
First 20:
['A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4', 'A4']


In [ ]:
from music21 import stream, note, chord

output_notes = []

for pattern in prediction_output:
    if "." in pattern:
        chord_notes = [int(n) for n in pattern.split(".")]
        output_notes.append(chord.Chord(chord_notes))
    else:
        output_notes.append(note.Note(pattern))

midi_stream = stream.Stream(output_notes)

output_path = project_path + "/generated_music/generated_music.mid"
midi_stream.write("midi", fp=output_path)

print("MIDI file created successfully!")
print(output_path)

MIDI file created successfully!
/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.mid


In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.mid"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
!apt-get -qq update
!apt-get -qq install -y fluidsynth
!pip -q install midi2audio

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Extracting templates from packages: 100%
Selecting previously unselected package libevdev2:amd64.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack .../00-libevdev2_1.13.1+dfsg-1build1_amd64.deb ...
Unpacking libevdev2:amd64 (1.13.1+dfsg-1build1) ...
Selecting previously unselected package at-spi2-common.
Preparing to unpack .../01-at-spi2-common_2.52.0-1build1_all.deb ...
Unpacking at-spi2-common (2.52.0-1build1) ...
Selecting previously unselected package libatspi2.0-0t64:amd64.
Preparing to unpack .../02-libatspi2.0-0t64_2.52.0-1build1_amd64.deb ...
Unpacking libatspi2.0-0t64:amd64 (2.52.0-1build1) ...
Selecting previously unselected package libxtst6:amd64.
Preparing to unpack .../03-libxtst6_2%3a1.2.3-1.1build1_amd64.deb ...
Unpacking libxtst6:amd64 (2:1

In [ ]:
!fluidsynth -ni /usr/share/sounds/sf2/default-GM.sf2 \
"/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.mid" \
-F "/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav" \
-r 44100

print("Audio file created successfully!")
print("/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav")

FluidSynth runtime version 2.3.4
Copyright (C) 2000-2023 Peter Hanappe and others.
Distributed under the LGPL license.
SoundFont(R) is a registered trademark of Creative Technology Ltd.

Rendering audio to file '/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav'..
Audio file created successfully!
/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav


In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
from google.colab import files

files.download(
    "/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav"
)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

wav_path = "/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav"

print("WAV exists:", os.path.exists(wav_path))

if os.path.exists(wav_path):
    print("File size:", os.path.getsize(wav_path), "bytes")

WAV exists: True
File size: 4905004 bytes


In [ ]:
from google.colab import files

files.download("/content/drive/MyDrive/AI_Music_Generation/generated_music/generated_music.wav")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import os

folder = "/content/drive/MyDrive/AI_Music_Generation/generated_music"

print(os.listdir(folder))

['generated_music.mid', 'generated_music.wav']
